In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm


def create_lags(y, p):
    df = pd.DataFrame({"y": y})
    for i in range(1, p + 1):
        df[f"lag_{i}"] = df["y"].shift(i)
    df = df.dropna()
    return df


def select_p(y, max_p, criterion):
    results = []
    df = create_lags(y, max_p) ### this ensures that the dataset that we use to select p is standardised for all p
    Y = df["y"]

    for p in range(1, max_p + 1):
        Y = df["y"]
        X_train = df[[f"lag_{i}" for i in range(1, p + 1)]]
        X_train = sm.add_constant(X_train, has_constant='add')

        model = sm.OLS(Y, X_train).fit()

        results.append({
            "p": p,
            "aic": model.aic,
            "bic": model.bic
        })

    results_df = pd.DataFrame(results)

    if criterion == "aic":
        best_p = results_df.loc[results_df["aic"].idxmin(), "p"]
    else:
        best_p = results_df.loc[results_df["bic"].idxmin(), "p"]

    return int(best_p), results_df


def ar_model_nowcast(y, max_p=8, criterion="bic"):
    # Step 1: select p
    best_p, ic_table = select_p(y, max_p, criterion)

    # Step 2: create lagged data
    df = create_lags(y, best_p)

    # Step 3: train-test split (last obs = test)
    train_df = df.iloc[:-1]
    test_df = df.iloc[-1:]

    y_train = train_df["y"]
    X_train = train_df.drop(columns=["y"])
    X_train = sm.add_constant(X_train, has_constant='add')

    y_test_actual = test_df["y"].values[0]
    X_test = test_df.drop(columns=["y"])
    X_test = sm.add_constant(X_test, has_constant='add')

    # Step 4: fit model
    model = sm.OLS(y_train, X_train).fit()

    # Step 5: predictions
    y_train_predicted = model.predict(X_train)
    y_test_predicted = model.predict(X_test).values[0]

    # Step 6: coefficients + SE
    coef = model.params
    se = model.bse

    coef_df = pd.DataFrame([coef, se])
    coef_df.index = ["coef", "se"]

    return {
        "best_p": best_p,
        "ic_table": ic_table,
        "coefficients": coef_df,
        "X": X_train,
        "y_actual": y_train.values,
        "y_train_predicted": y_train_predicted.values,
        "y_test_actual": y_test_actual,
        "y_test_predicted": y_test_predicted
    }

In [ ]:
# Example GDP series
df = pd.read_excel("../mock_data/GDPPCT.xlsx", sheet_name ="Quarterly")
df = df[(df["observation_date"].dt.year < 2020) & (df["observation_date"].dt.year >= 1959)]
y = df["A191RL1Q225SBEA"]
result = ar_model_nowcast(y, max_p=5, criterion="aic")

print("Best p:", result["best_p"])
print(result["coefficients"])
print("Test actual:", result["y_test_actual"])
print("Test predicted:", result["y_test_predicted"])

Best p: 2
         const     lag_1     lag_2
coef  1.800062  0.233770  0.189950
se    0.312648  0.064263  0.064356
Test actual: 7.9
Test predicted: 4.0311077647695


In [5]:
select_p(y, max_p=5, criterion="bic")

(2,
    p          aic          bic
 0  1  1242.495887  1249.448814
 1  2  1236.400910  1246.830301
 2  3  1238.240374  1252.146228
 3  4  1239.842475  1257.224793
 4  5  1239.328081  1260.186862)